Below I’ll give you modular PyTorch functions for:

* Real Möbius map ($M_w(x)$) on ball / sphere
* Kuramoto-on-sphere order parameter and vector field
* Reduced hyperbolic ($w$)-dynamics (both via Möbius and via autograd gradient)
* Simple simulation + Plotly visualization hooks for 2D (disk/$S^1$) and 3D (ball/$S^2$)

Everything is real-valued, dot-product based, and autograd-friendly.

# 1. Core math building blocks (PyTorch)

In [1]:
import torch
from torch import Tensor
import math

## 1.1 Basic helpers

In [2]:
def normalize(x: Tensor, eps: float = 1e-9) -> Tensor:
    """
    Normalize vectors along the last dimension.
    x: [..., d]
    returns: [..., d] with unit norm (or zero if input ~0).
    """
    norm = x.norm(dim=-1, keepdim=True)
    return x / (norm + eps)


def dot(a: Tensor, b: Tensor) -> Tensor:
    """
    Dot product along the last dimension.
    a, b: [..., d]
    returns: [...] (scalar per batch entry)
    """
    return (a * b).sum(dim=-1)

## 1.2 Real Möbius map on ball / sphere
This is the real version of the Möbius map ($M_w$) we discussed:

$$
M_w(x) = \frac{(1 - |w|^2)(x - |x|^2 w)}{1 - 2\langle w,x\rangle + |w|^2 |x|^2} - w.
$$

In [3]:
from LMS import (
    random_points_on_sphere, integrate_w_euler, make_sphere_figure, reconstruct_sphere_trajectory
)
import torch

p3 = random_points_on_sphere(640, 3)
a = torch.ones(640)
res = integrate_w_euler(torch.zeros(3), p3, a, dt=1e-3, steps=300)
x_traj = reconstruct_sphere_trajectory(res.trajectory, p3)
fig = make_sphere_figure(x_traj[-1], w_t=res.trajectory[-1], w_path=res.trajectory)
fig.show()


In [4]:
from LMS import make_lms_circle_plotly_widget

widget = make_lms_circle_plotly_widget(steps=420, dt=0.012, rng_seed=0)
display(widget.layout)



    'data': [{'hoverinfo': 'skip',
              'line': {'color': 'black', 'wid…

In [5]:
from lms_plotly_widget import LMSCirclePlotlyWidget, SliderSpec

widget = LMSCirclePlotlyWidget(
    slider_specs=[
        SliderSpec("custom_alpha", "Custom alpha", 0.0, 1.0, 0.05, 0.5),
    ]
)
display(widget.layout)


    'data': [{'hoverinfo': 'skip',
              'line': {'color': 'black', 'wid…

In [6]:
from LMS import make_lms_ball3d_widget
from IPython.display import display

widget = make_lms_ball3d_widget(w_mode="autograd", rng_seed=0)
display(widget.layout)

import torch
from LMS import (
    random_points_on_sphere, skew_symmetric_from_axis, integrate_lms_reduced_euler
)

N, d = 128, 3
base = random_points_on_sphere(N, d=d, dtype=torch.float64)
weights = torch.ones(N, dtype=torch.float64) / N
w0 = torch.tensor([0.05, 0.0, 0.0], dtype=torch.float64)
zeta0 = torch.eye(3, dtype=torch.float64)
A = skew_symmetric_from_axis(torch.tensor([0.,0.,1.], dtype=torch.float64), rate=1.0)

traj = integrate_lms_reduced_euler(
    w0, zeta0, base, weights, A=A, coupling=1.0, dt=5e-2, steps=1000, w_mode="autograd"
)


Box(children=(FigureWidget({
    'data': [{'hoverinfo': 'skip',
              'line': {'color': 'lightgrey', '…

In [7]:
from LMS import make_lms_ball3d_backward_two_sheet_widget
from IPython.display import display

widget = make_lms_ball3d_backward_two_sheet_widget(
    w_mode="autograd",
    rng_seed=0,
    outer_radius_display=2.5,   # camera range for outside sheet
    
    outer_radius_cap=1.0,       # clip very large radii for readability
    force_backward_time=True,   # native backward dynamics
)
display(widget.layout)


Box(children=(FigureWidget({
    'data': [{'hoverinfo': 'skip',
              'line': {'color': 'lightgrey', '…

In [8]:
#@title AdamW
# # import matplotlib.pyplot as plt
# import networkx as nx
# from matplotlib.patches import FancyArrowPatch, Rectangle
# import math

# # ============================================================
# # Goal: Two causal diagrams that make "Möbius inversion kills
# # accumulated influence" visually obvious.
# #
# # Graph A (Regular / Zeta-memory):
# #   innovation u_k feeds an accumulator A_t = (zeta_alpha v)_t
# #   so early shocks have long-range influence via A_t -> x_t.
# #
# # Graph B (Local / Möbius-inverted):
# #   eliminate A_t and show the local inverse:
# #     v_t = (alpha * x_{t-1} - x_t)/eta
# #   so influence is only adjacent in time.
# # ============================================================

# # ---------- shared rendering utilities ----------
# def draw_fancy_edge(ax, pos, u, v, label="", rad=0.0, label_pos=0.5,
#                     color="black", style="solid", lw=1.6, alpha=0.85):
#     if u not in pos or v not in pos:
#         return
#     (x1, y1), (x2, y2) = pos[u], pos[v]
#     patch = FancyArrowPatch(
#         (x1, y1), (x2, y2),
#         connectionstyle=f"arc3,rad={rad}",
#         arrowstyle='-|>',
#         mutation_scale=18,
#         linewidth=lw,
#         color=color,
#         linestyle=style,
#         shrinkA=14, shrinkB=14,
#         alpha=alpha
#     )
#     ax.add_patch(patch)

#     if not label:
#         return

#     dx, dy = x2 - x1, y2 - y1
#     dist = math.hypot(dx, dy)
#     nx_vec, ny_vec = -dy, dx
#     if dist > 0:
#         nx_vec /= dist
#         ny_vec /= dist

#     bx = x1 + dx * label_pos
#     by = y1 + dy * label_pos
#     arc_height = rad * (dist / 2.0)
#     lx = bx + nx_vec * arc_height
#     ly = by + ny_vec * arc_height

#     ax.text(
#         lx, ly, label,
#         fontsize=10, color=color,
#         ha="center", va="center",
#         bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.92),
#         zorder=10
#     )


# def draw_graph(title, nodes, pos, edges, boxes=None, xlim=None, ylim=None, figsize=(16, 6.5)):
#     fig, ax = plt.subplots(figsize=figsize)

#     # nodes
#     G = nx.DiGraph()
#     G.add_nodes_from(nodes)

#     nx.draw_networkx_nodes(
#         G, pos, node_size=2200, node_color="white",
#         edgecolors="black", linewidths=1.5, ax=ax
#     )
#     nx.draw_networkx_labels(G, pos, font_size=12, font_weight="bold", ax=ax)

#     # optional boxes (for "accumulator" operator)
#     if boxes:
#         for b in boxes:
#             # b: dict with x,y,w,h,label,color,alpha
#             rect = Rectangle(
#                 (b["x"], b["y"]), b["w"], b["h"],
#                 linewidth=1.8,
#                 edgecolor=b.get("color", "black"),
#                 facecolor="none",
#                 linestyle=b.get("style", "solid"),
#                 alpha=b.get("alpha", 0.9),
#                 zorder=2
#             )
#             ax.add_patch(rect)
#             ax.text(
#                 b["x"] + b["w"]/2, b["y"] + b["h"] + 0.35,
#                 b.get("label",""),
#                 fontsize=12,
#                 color=b.get("color", "black"),
#                 ha="center", va="bottom"
#             )

#     # edges
#     for e in edges:
#         draw_fancy_edge(
#             ax, pos,
#             e["u"], e["v"],
#             label=e.get("label",""),
#             rad=e.get("rad",0.0),
#             label_pos=e.get("pos",0.5),
#             color=e.get("color","black"),
#             style=e.get("style","solid"),
#             lw=e.get("lw",1.6),
#             alpha=e.get("alpha",0.85)
#         )

#     ax.set_title(title, fontsize=16, pad=15)
#     ax.set_axis_off()
#     if xlim: ax.set_xlim(*xlim)
#     if ylim: ax.set_ylim(*ylim)
#     plt.tight_layout()
#     plt.show()


# # ---------- helpers for node names ----------
# def N(var, t):  # var is already short: x,v,u,A,alpha,g
#     return f"{var}[{t}]"


# # ============================================================
# # Graph A: Regular form (Zeta-memory explicit)
# # ============================================================
# def graph_zeta_memory():
#     times = ["0", "1", "2"]

#     # minimal variable set to make the point
#     vars_ = ["x", "g", "u", "v", "A", "α"]  # A is the accumulator, α is WD factor

#     nodes = [N(v,t) for t in times for v in vars_]

#     # layout: 3 columns; x on top; g/u/v middle; A boxed low; α under x
#     base_x = {"0": -10.0, "1": 0.0, "2": 10.0}
#     x_off  = {"x":  0.0, "g": -2.6, "u": -2.6, "v":  2.6, "A":  0.0, "α":  0.0}
#     y_map  = {"x":  8.0, "g":  6.5, "u":  5.0, "v":  5.7, "A":  2.6, "α":  3.7}

#     pos = {N(v,t):(base_x[t]+x_off[v], y_map[v]) for t in times for v in vars_}

#     edges = []

#     # state -> gradient -> innovation -> velocity
#     edges += [
#         dict(u=N("x","0"), v=N("g","0"), label=r"$\nabla y$", rad=0.22, color="#777"),
#         dict(u=N("g","0"), v=N("u","0"), label=r"$\phi$", rad=0.00, color="black"),
#         dict(u=N("u","0"), v=N("v","1"), label=r"to $v_{1}$", rad=0.10, color="black"),

#         dict(u=N("x","1"), v=N("g","1"), label=r"$\nabla y$", rad=0.22, color="#777"),
#         dict(u=N("g","1"), v=N("u","1"), label=r"$\phi$", rad=0.00, color="black"),
#         dict(u=N("u","1"), v=N("v","2"), label=r"to $v_{2}$", rad=0.10, color="black"),
#     ]

#     # accumulator definition: A_t = sum_{k<=t} alpha^{t-k} v_k (show only a few arrows)
#     # Draw v1 -> A1, v1 -> A2, v2 -> A2, plus a faint "shock persists" arrow u0 -> A2
#     edges += [
#         dict(u=N("v","1"), v=N("A","1"), label=r"$1$", rad=-0.10, color="#444"),
#         dict(u=N("v","1"), v=N("A","2"), label=r"$\alpha$", rad=0.18, color="#999", style="dotted"),
#         dict(u=N("v","2"), v=N("A","2"), label=r"$1$", rad=-0.10, color="#444"),

#         # show that a shock at u0 affects A2 indirectly (the "long memory" vibe)
#         dict(u=N("u","0"), v=N("A","2"), label=r"shock persists", rad=0.30,
#              color="#999", style="dotted", alpha=0.7),
#     ]

#     # A_t -> x_t (state depends on accumulated update)
#     edges += [
#         dict(u=N("A","1"), v=N("x","1"), label=r"$-\eta$", rad=0.18, color="black"),
#         dict(u=N("A","2"), v=N("x","2"), label=r"$-\eta$", rad=0.18, color="black"),
#     ]

#     # WD / contraction path: x_{t-1} -> x_t with coefficient alpha
#     edges += [
#         dict(u=N("α","0"), v=N("x","1"), label=r"$\alpha=1-\eta\lambda$", rad=-0.35,
#              color="red", style="dashed"),
#         dict(u=N("x","0"), v=N("x","1"), label=r"$\times\alpha$", rad=-0.12, color="red"),

#         dict(u=N("α","1"), v=N("x","2"), label=r"$\alpha$", rad=-0.35,
#              color="red", style="dashed"),
#         dict(u=N("x","1"), v=N("x","2"), label=r"$\times\alpha$", rad=-0.12, color="red"),
#     ]

#     # residual "unexplained" gray arrow to suggest: x0 influences x2 by passing through the accumulator
#     edges += [
#         dict(u=N("x","0"), v=N("x","2"), label=r"via memory", rad=0.40,
#              color="#bbb", style="dotted", alpha=0.6)
#     ]

#     # accumulator boxes: draw one big operator box spanning A[0],A[1],A[2]
#     # We'll compute the rectangle bounds from the node positions.
#     xs = [pos[N("A",t)][0] for t in times]
#     ys = [pos[N("A",t)][1] for t in times]
#     box = dict(
#         x=min(xs)-3.3, y=min(ys)-0.8,
#         w=(max(xs)-min(xs))+6.6, h=(max(ys)-min(ys))+1.6,
#         label=r"Zeta / accumulation:  $A_t = (\zeta_\alpha v)_t$",
#         color="#444", style="solid", alpha=0.9
#     )

#     draw_graph(
#         title="A) Regular (Zeta-memory): early influence accumulates and persists through A_t",
#         nodes=nodes, pos=pos, edges=edges, boxes=[box],
#         xlim=(-14, 14), ylim=(1.2, 9.4)
#     )


# # ============================================================
# # Graph B: Local Möbius-inverted form (no accumulator)
# # ============================================================
# def graph_mobius_local():
#     times = ["0", "1", "2"]
#     vars_ = ["x", "g", "u", "v", "α"]  # no A node

#     nodes = [N(v,t) for t in times for v in vars_]

#     base_x = {"0": -10.0, "1": 0.0, "2": 10.0}
#     x_off  = {"x":  0.0, "g": -2.6, "u": -2.6, "v":  2.6, "α":  0.0}
#     y_map  = {"x":  8.0, "g":  6.5, "u":  5.0, "v":  5.7, "α":  3.7}

#     pos = {N(v,t):(base_x[t]+x_off[v], y_map[v]) for t in times for v in vars_}

#     edges = []

#     # Keep the same "learning mechanism" chain (x -> g -> u -> v_next) so it's comparable
#     edges += [
#         dict(u=N("x","0"), v=N("g","0"), label=r"$\nabla y$", rad=0.22, color="#777"),
#         dict(u=N("g","0"), v=N("u","0"), label=r"$\phi$", rad=0.00, color="black"),
#         dict(u=N("u","0"), v=N("v","1"), label=r"to $v_{1}$", rad=0.10, color="black"),

#         dict(u=N("x","1"), v=N("g","1"), label=r"$\nabla y$", rad=0.22, color="#777"),
#         dict(u=N("g","1"), v=N("u","1"), label=r"$\phi$", rad=0.00, color="black"),
#         dict(u=N("u","1"), v=N("v","2"), label=r"to $v_{2}$", rad=0.10, color="black"),
#     ]

#     # The Möbius-local statement: v_t depends only on (x_{t-1}, x_t, alpha) locally
#     # v_t = (alpha x_{t-1} - x_t)/eta
#     edges += [
#         dict(u=N("α","0"), v=N("v","1"), label=r"$\alpha$", rad=-0.30, color="red", style="dashed"),
#         dict(u=N("x","0"), v=N("v","1"), label=r"$+\alpha/\eta$", rad=-0.10, color="red"),
#         dict(u=N("x","1"), v=N("v","1"), label=r"$-1/\eta$", rad=0.10, color="black"),

#         dict(u=N("α","1"), v=N("v","2"), label=r"$\alpha$", rad=-0.30, color="red", style="dashed"),
#         dict(u=N("x","1"), v=N("v","2"), label=r"$+\alpha/\eta$", rad=-0.10, color="red"),
#         dict(u=N("x","2"), v=N("v","2"), label=r"$-1/\eta$", rad=0.10, color="black"),
#     ]

#     # Forward dynamics (optional, faint): show that v_t still updates x_t (but this is "redundant" in the inverted view)
#     edges += [
#         dict(u=N("v","1"), v=N("x","1"), label=r"$-\eta$", rad=0.25, color="#888", style="dotted"),
#         dict(u=N("v","2"), v=N("x","2"), label=r"$-\eta$", rad=0.25, color="#888", style="dotted"),
#         dict(u=N("x","0"), v=N("x","1"), label=r"$\times\alpha$", rad=-0.12, color="red", alpha=0.35),
#         dict(u=N("x","1"), v=N("x","2"), label=r"$\times\alpha$", rad=-0.12, color="red", alpha=0.35),
#     ]

#     # The "killed accumulated influence" visual: DO NOT draw any long arrow like u0 -> v2 or x0 -> v2.
#     # Instead, add an explicit crossed-out annotation as text (minimal, but readable).
#     # We'll place it near the center.
#     # (Keeping it as text avoids messy "no-edge" symbolism.)
#     # We'll use axes coordinates for stable placement.
#     # Note: this is not a node; it's an annotation.
#     fig, ax = plt.subplots(figsize=(16, 6.5))

#     # build and draw nodes/labels manually here so we can add the annotation cleanly
#     G = nx.DiGraph()
#     G.add_nodes_from(nodes)
#     nx.draw_networkx_nodes(
#         G, pos, node_size=2200, node_color="white",
#         edgecolors="black", linewidths=1.5, ax=ax
#     )
#     nx.draw_networkx_labels(G, pos, font_size=12, font_weight="bold", ax=ax)

#     for e in edges:
#         draw_fancy_edge(
#             ax, pos, e["u"], e["v"],
#             label=e.get("label",""),
#             rad=e.get("rad",0.0),
#             label_pos=e.get("pos",0.5),
#             color=e.get("color","black"),
#             style=e.get("style","solid"),
#             lw=e.get("lw",1.6),
#             alpha=e.get("alpha",0.85)
#         )

#     ax.text(
#         0.5, 0.07,
#         r"Möbius inversion makes the dependency local:  $v_t \Leftarrow (x_{t-1},x_t,\alpha)$  (no long-range arrows)",
#         transform=ax.transAxes, ha="center", va="center",
#         fontsize=12,
#         bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="black", alpha=0.9)
#     )

#     ax.set_title("B) Möbius-local: accumulated influence collapses to a 2-slice Markov blanket", fontsize=16, pad=15)
#     ax.set_axis_off()
#     ax.set_xlim(-14, 14)
#     ax.set_ylim(1.2, 9.4)
#     plt.tight_layout()
#     plt.show()


# # ---------- run both ----------
# graph_zeta_memory()
# graph_mobius_local()

In [9]:
from LMS import make_lms_ball3d_hydrodynamic_ensemble_widget
w = make_lms_ball3d_hydrodynamic_ensemble_widget()
w.layout


Box(children=(FigureWidget({
    'data': [{'hoverinfo': 'skip',
              'line': {'color': 'lightgrey', '…